In [18]:
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [20]:
df = pd.read_csv(r"D:\Summer_2025_ML_Internships\Developer_HUB_ML_Tasks\Advanced\End_to_End_ML_Pipeline\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.tail(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


In [21]:

df = df.replace(" ", np.nan)
df.drop_duplicates()
df.dropna(axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [22]:
df["Churn"] = df["Churn"].astype(str).str.strip()
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [23]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), X.select_dtypes(include=["int64", "float64"]).columns),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), X.select_dtypes(include=["object"]).columns ),
    ],
    remainder="passthrough"
)

In [30]:
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500))
])

model_log_reg = log_reg.fit(X_train, y_train)

rf_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model_rf_clf = rf_clf.fit(X_train, y_train)

In [28]:
lg_param_grid = [
    # L1 penalty only works with liblinear or saga
    {
        "classifier__penalty": ["l1"],
        "classifier__C": np.logspace(-4, 4, 13),
        "classifier__solver": ["liblinear", "saga"],
        "classifier__max_iter": [100, 250, 500, 1000]
    },
    # L2 penalty works with liblinear, saga, newton-cg, sag, lbfgs
    {
        "classifier__penalty": ["l2"],
        "classifier__C": np.logspace(-4, 4, 13),
        "classifier__solver": ["liblinear", "saga", "newton-cg", "sag", "lbfgs"],
        "classifier__max_iter": [100, 250, 500, 1000]
    },
    # Elasticnet penalty only support saga, must include l1_ratio
    {
        "classifier__penalty": ["elasticnet"],
        "classifier__C": np.logspace(-4, 4, 13),
        "classifier__solver": ["saga"],
        "classifier__l1_ratio": [0.2, 0.5, 0.8],
        "classifier__max_iter": [100, 250, 500, 1000]
    },
    # No penalty
    {
        "classifier__penalty": [None],
        "classifier__C": [1.0],
        "classifier__solver": ["saga", "lbfgs"],
        "classifier__max_iter": [100, 250, 500, 1000]
    }
]


gridsearch_log = GridSearchCV(log_reg, param_grid= lg_param_grid, cv= 3, verbose=True,n_jobs=-1)
gridsearch_log.fit(X_train, y_train)
print("Best Parameters: ", gridsearch_log.best_params_)

Fitting 3 folds for each of 528 candidates, totalling 1584 fits
Best Parameters:  {'classifier__C': np.float64(21.54434690031882), 'classifier__max_iter': 250, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}


In [33]:
rf_param_grid = {
    'classifier__n_estimators': [200, 500],
    'classifier__max_features': ['sqrt', 'log2'],
    'classifier__max_depth' : [4,5,6,7,8],
    'classifier__criterion' :['gini', 'entropy']
}

gridsearch_randfc = GridSearchCV(estimator=rf_clf, param_grid= rf_param_grid, cv= 3)
gridsearch_randfc.fit(X_train, y_train)

print("Best Parameters: ", gridsearch_randfc.best_params_)

Best Parameters:  {'classifier__criterion': 'gini', 'classifier__max_depth': 4, 'classifier__max_features': 'sqrt', 'classifier__n_estimators': 200}


In [34]:
models = {
    "Logistic Regression": gridsearch_log,
    "Random Forest": gridsearch_randfc
}
for name, model in models.items():
    print(f"\n=== {name} Evaluation ===")
    y_pred = model.predict(X_test)
    print("Accuracy Score: ", accuracy_score(y_test, y_pred))


=== Logistic Regression Evaluation ===
Accuracy Score:  0.8190205819730305

=== Random Forest Evaluation ===
Accuracy Score:  0.7352732434350603


In [35]:
import joblib

joblib.dump(gridsearch_log, "logistic_regression_churn.pkl")
print("Logistic Regression model saved as logistic_regression_churn.pkl")

joblib.dump(gridsearch_randfc, "random_forest_churn.pkl")
print("Random Forest model saved as random_forest_churn.pkl")

Logistic Regression model saved as logistic_regression_churn.pkl
Random Forest model saved as random_forest_churn.pkl
